# Lab type: review
# Course: ML201 — Applied Machine Learning
# Lesson: Model Calibration and Probability Estimation
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 3000

age = np.random.uniform(30, 70, n)
bmi = np.random.normal(27, 5, n).clip(15, 50)
blood_pressure = np.random.normal(120, 15, n).clip(80, 180)
cholesterol = np.random.normal(200, 35, n).clip(120, 320)
family_history = np.random.binomial(1, 0.3, n)
smoking_years = np.random.exponential(5, n).clip(0, 40)
exercise_frequency = np.random.randint(0, 6, n)

log_odds = (
    -1.5
    + 0.04 * (age - 50)
    + 0.06 * (bmi - 27)
    + 0.02 * (blood_pressure - 120)
    + 0.01 * (cholesterol - 200)
    + 0.8 * family_history
    + 0.05 * smoking_years
    - 0.15 * exercise_frequency
)
prob_disease = 1 / (1 + np.exp(-log_odds))
disease = np.random.binomial(1, prob_disease)

df = pd.DataFrame({
    'age': age,
    'bmi': bmi,
    'blood_pressure': blood_pressure,
    'cholesterol': cholesterol,
    'family_history': family_history,
    'smoking_years': smoking_years,
    'exercise_frequency': exercise_frequency,
    'disease': disease
})

print(f"Dataset shape: {df.shape}")
print(f"Positive rate: {df['disease'].mean():.2%}")

X = df.drop('disease', axis=1)
y = df['disease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lr = LogisticRegression(max_iter=1000, random_state=42)
gbm = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42)

lr.fit(X_train, y_train)
gbm.fit(X_train, y_train)
rf.fit(X_train, y_train)

print("\nTest AUC:")
for name, model in [('LogisticRegression', lr), ('GradientBoosting', gbm), ('RandomForest', rf)]:
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f"  {name}: {auc:.3f}")

## Part 1: Reliability Diagrams

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

models = [
    ('LogisticRegression', lr, 'steelblue'),
    ('GradientBoosting', gbm, 'tomato'),
    ('RandomForest', rf, 'seagreen'),
]

ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')

for name, model, color in models:
    probs = model.predict_proba(X_test)[:, 1]
    fraction_of_positives, mean_predicted_value = calibration_curve(y_test, probs, n_bins=10, strategy='quantile')
    ax.plot(mean_predicted_value, fraction_of_positives, marker='o', label=name, color=color)

ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Reliability Diagrams — all three models')
ax.legend()
plt.tight_layout()
plt.show()

**Question 1:** The GradientBoosting curve shows an S-shape — low predictions are too low, high predictions are too high. This is called "overconfidence." What mechanism in tree-based models causes this? *(Think about what is stored in the leaves.)*

*(Write your answer here.)*

**Question 2:** LogisticRegression sits close to the diagonal. Why is logistic regression naturally well-calibrated, whereas tree ensembles are not?

*(Write your answer here.)*

## Part 2: Calibrating with CalibratedClassifierCV

In [ ]:
calibrated_sigmoid = CalibratedClassifierCV(
    GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
    method='sigmoid', cv=5
)
calibrated_isotonic = CalibratedClassifierCV(
    GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
    method='isotonic', cv=5
)

calibrated_sigmoid.fit(X_train, y_train)
calibrated_isotonic.fit(X_train, y_train)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

variants = [
    ('Uncalibrated GBM', gbm),
    ('Platt-scaled (sigmoid)', calibrated_sigmoid),
    ('Isotonic regression', calibrated_isotonic),
]

for ax, (label, model) in zip(axes, variants):
    probs = model.predict_proba(X_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10, strategy='quantile')
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.plot(mean_pred, frac_pos, marker='o', color='tomato', label=label)
    ax.set_title(label)
    ax.set_xlabel('Mean predicted probability')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Fraction of positives')
plt.suptitle('Calibration: Uncalibrated vs Platt vs Isotonic', fontsize=13)
plt.tight_layout()
plt.show()

**Question 3:** Platt scaling (sigmoid) fits a single sigmoid function to the raw outputs. Isotonic regression fits a piecewise monotone function. On a calibration set of only 200 examples, which method would you prefer and why?

*(Write your answer here.)*

**Question 4:** `cv=5` in `CalibratedClassifierCV` means the underlying model is trained 5 times, each time calibrated on a held-out fold. What would happen if you used `cv='prefit'` on a model already trained on the full training set? When is `'prefit'` appropriate?

*(Write your answer here.)*

## Part 3: When Calibration Matters

In [ ]:
# --- Scenario 1: Ranking (precision@50) ---
probs_uncal = gbm.predict_proba(X_test)[:, 1]
probs_cal = calibrated_isotonic.predict_proba(X_test)[:, 1]
y_test_arr = y_test.values

def precision_at_k(probs, y_true, k=50):
    top_k_idx = np.argsort(probs)[::-1][:k]
    return y_true[top_k_idx].mean()

p50_uncal = precision_at_k(probs_uncal, y_test_arr, k=50)
p50_cal = precision_at_k(probs_cal, y_test_arr, k=50)

print("=== Scenario 1: Ranking (Precision@50) ===")
print(f"  Uncalibrated GBM : {p50_uncal:.3f}")
print(f"  Calibrated (iso) : {p50_cal:.3f}")

# --- Scenario 2: Threshold decision at p=0.30 ---
threshold = 0.30
flagged_uncal = (probs_uncal >= threshold).sum()
flagged_cal = (probs_cal >= threshold).sum()

print("\n=== Scenario 2: Decision at threshold = 0.30 ===")
print(f"  Patients flagged (uncalibrated) : {flagged_uncal}")
print(f"  Patients flagged (calibrated)   : {flagged_cal}")
print(f"  Difference                      : {abs(flagged_cal - flagged_uncal)} patients")

**Question 5:** For ranking tasks (recommend top-N users), does calibration improve precision@50? Why or why not?

*(Write your answer here.)*

**Question 6:** A doctor uses the model to flag patients for additional testing. The decision rule is "order tests if predicted probability > 0.15." The uncalibrated GBM outputs 0.08 for a patient whose actual risk (per logistic regression) is 0.22. What is the clinical consequence of miscalibration in this case?

*(Write your answer here.)*

## Summary

Answer these final check questions in one sentence each:

1. A reliability diagram plots mean predicted probability on the x-axis and fraction of positives on the y-axis. What does a point far above the diagonal tell you about those predictions?

2. You have a large calibration holdout set (n=5000). Which calibration method — Platt scaling or isotonic regression — is likely to produce a better-calibrated model, and why does sample size matter for this choice?

3. A model will be used only to rank leads for a sales team (top 100 calls per week). Does calibration matter for this deployment? Explain in one sentence.